In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline


In [2]:
words = open('names.txt', 'r').read().splitlines()
words[:5]
len(words)

32033

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [13]:
# construct the dataset
block_size = 3
X, Y = [], []
for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]
X = torch.tensor(X)
Y = torch.tensor(Y)

In [14]:
X.shape

torch.Size([228146, 3])

In [15]:
Y.shape

torch.Size([228146])

In [7]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]


In [8]:
sum(p.nelement() for p in parameters)

3481

In [9]:
for p in parameters:
    p.requires_grad = True

In [ ]:
# Forward pass
  # embed the characters into vectors
for i in range(1000):
    # construct the mini-batch
    ix =torch.randint(0, X.shape[0], (32,))
    emb = C[X[ix]]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2

    loss = F.cross_entropy(logits, Y[ix])
    for p in parameters:
        p.grad = None  # set the gradients to zero
    # print(loss.item())
    loss.backward()

    for p in parameters:
        p.data += -0.15 * p.grad  # update the parameters with gradient descent

print(loss.item())


2.0371241569519043


In [24]:
emb = C[X]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2

loss = F.cross_entropy(logits, Y)
loss

tensor(2.5579, grad_fn=<NllLossBackward0>)